# Vast AI — End-to-End Pipeline Test

Tests the full pipeline **without training**: data loading, preprocessing, all `.pth` model checkpoints, and the gender LR model.

| Section | What it tests |
|---|---|
| 1 | Data loading, taxonomy merge, label encoding, datasets, dataloaders |
| 2a | CLIP Hierarchical (`outputs/clip_hierarchical_best.pth`) |
| 2b | DINOv2 Hierarchical (`outputs/dino_best.pth`) |
| 2c | CLIP Multitask (`outputs/clip_multitask_best.pth`) |
| 2d | EfficientNet-B3 Hierarchical (`EfficientNet/efficientnet_b3_best.pth`) |
| 2e | ResNet18 Color (`outputs/color_resnet18_best.pth`) |
| 2f | Gender Logistic Regression (`outputs/gender_lr_model.pkl`) |
| 3 | CLIP + DINOv2 Ensemble inference |
| 4 | Results summary table |

## 0. Imports & Setup

In [ ]:
import os
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from transformers import CLIPModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Config & Paths

All paths are relative to this notebook's location (project root).

In [ ]:
# ── Directories ───────────────────────────────────────────────────────────────
TRAIN_DIR      = './Data/train'
DISC_DIR       = './Data/discovery'
OUTPUT_DIR     = './outputs'
IMAGE_DIR      = './Data/train/images'
DISC_IMAGE_DIR = './Data/discovery/images'

# ── Data files ────────────────────────────────────────────────────────────────
PARQUET_PATH  = './Data/train/train.parquet'
DISC_PARQUET  = './Data/discovery/discovery.parquet'
TAXONOMY_PATH = './Data/level2_categories.json'   # lives at project root Data/

# ── Model checkpoints (.pth) ──────────────────────────────────────────────────
CLIP_HIER_PATH  = './outputs/clip_hierarchical_best.pth'
CLIP_MT_PATH    = './outputs/clip_multitask_best.pth'
DINO_PATH       = './outputs/dino_best.pth'
COLOR_PATH      = './outputs/color_resnet18_best.pth'
EFFNET_PATH     = './EfficientNet/efficientnet_b3_best.pth'

# ── Sklearn artifacts (.pkl) ──────────────────────────────────────────────────
CAT_ENC_PATH    = './outputs/category_encoder.pkl'
COLOR_ENC_PATH  = './outputs/color_encoder.pkl'
GENDER_ENC_PATH = './outputs/gender_label_encoder.pkl'
GENDER_LR_PATH  = './outputs/gender_lr_model.pkl'
GENDER_COLS_PATH= './outputs/gender_feature_cols.pkl'

# ── Constants (match training) ────────────────────────────────────────────────
NUM_MAIN    = 21
NUM_SUB     = 190
NUM_COLORS  = 11   # non-unknown colors; full multitask uses 12 (includes unknown)
NUM_GENDERS = 4
IMG_SIZE    = 224
BATCH_SIZE  = 32   # smaller batch for testing

# Sanity-check key paths exist
for label, path in [
    ('train.parquet',  PARQUET_PATH),
    ('taxonomy',       TAXONOMY_PATH),
    ('image_dir',      IMAGE_DIR),
    ('clip_hier',      CLIP_HIER_PATH),
    ('clip_multitask', CLIP_MT_PATH),
    ('dino',           DINO_PATH),
    ('color_resnet',   COLOR_PATH),
    ('efficientnet',   EFFNET_PATH),
    ('gender_lr',      GENDER_LR_PATH),
]:
    status = '✓' if os.path.exists(path) else '✗ MISSING'
    print(f'  {status}  {label:20s}  {path}')

---
## Part 1 — Data Pipeline

### 1.1 Load Parquet Files

In [ ]:
train_df = pd.read_parquet(PARQUET_PATH)
print(f'train.parquet  →  {train_df.shape}')
print(f'Columns: {list(train_df.columns)}')

if os.path.exists(DISC_PARQUET):
    disc_df  = pd.read_parquet(DISC_PARQUET)
    df = pd.concat([train_df, disc_df]).reset_index(drop=True)
    print(f'\ndiscovery.parquet  →  {disc_df.shape}')
    print(f'Combined           →  {df.shape}')
else:
    df = train_df.copy()
    print('discovery.parquet not found — using train only')

print('\nFirst row:')
df.head(1)

### 1.2 Taxonomy Merge & Category Clean-up

In [ ]:
with open(TAXONOMY_PATH, 'r') as f:
    mapping = json.load(f)

taxonomy = pd.DataFrame(mapping).rename(columns={'new_id': 'category_id'})
df = df.merge(taxonomy[['category_id', 'category_name']], on='category_id', how='left')
df[['category', 'subcategory']] = df['category_name'].str.split(' > ', expand=True)

# Merge rare class 44 (2 samples) into class 28
n_cat44 = (df['category_id'] == 44).sum()
df.loc[df['category_id'] == 44, 'category_id'] = 28
print(f'Merged {n_cat44} rows from category_id=44 into 28')

# Fill missing values
df['gender']      = df['gender'].fillna('UNISEX')
df['color_label'] = df['color_label'].fillna('unknown')

print(f'\nShape: {df.shape}')
print(f'Subcategories: {df["category_id"].nunique()}')
print(f'Main categories: {df["category"].nunique()}')
print(f'Color classes: {df["color_label"].nunique()}')
print(f'Gender classes: {df["gender"].nunique()}')
df[['category_id','category','subcategory','color_label','gender']].head(3)

### 1.3 Label Encoding

In [ ]:
# Main category (21 classes)
main_encoder = LabelEncoder()
df['main_encoded'] = main_encoder.fit_transform(df['category'])

# Subcategory (190 classes)
cat_encoder = LabelEncoder()
df['category_encoded'] = cat_encoder.fit_transform(df['category_id'])

# Color (11 known + 1 unknown = 12 total; -1 for unknown in multitask)
df_with_color = df[df['color_label'] != 'unknown'].copy()
color_encoder = LabelEncoder()
df_with_color['color_encoded'] = color_encoder.fit_transform(df_with_color['color_label'])
df['color_encoded'] = -1
df.loc[df['color_label'] != 'unknown', 'color_encoded'] = color_encoder.transform(
    df.loc[df['color_label'] != 'unknown', 'color_label']
)

# Gender (4 classes)
gender_encoder = LabelEncoder()
df['gender_encoded'] = gender_encoder.fit_transform(df['gender'])

NUM_MAIN_     = df['main_encoded'].nunique()
NUM_SUB_      = df['category_encoded'].nunique()
NUM_COLORS_   = df_with_color['color_label'].nunique()
NUM_GENDERS_  = df['gender_encoded'].nunique()

print(f'Main categories : {NUM_MAIN_}   → {list(main_encoder.classes_)}')
print(f'Subcategories   : {NUM_SUB_}')
print(f'Color classes   : {NUM_COLORS_}  → {list(color_encoder.classes_)}')
print(f'Gender classes  : {NUM_GENDERS_}   → {list(gender_encoder.classes_)}')

# Update constants to match actual data
NUM_MAIN    = NUM_MAIN_
NUM_SUB     = NUM_SUB_
NUM_COLORS  = NUM_COLORS_
NUM_GENDERS = NUM_GENDERS_
print(f'\n[Updated] NUM_MAIN={NUM_MAIN}, NUM_SUB={NUM_SUB}, NUM_COLORS={NUM_COLORS}, NUM_GENDERS={NUM_GENDERS}')

### 1.4 Train / Test Split

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.1,
    stratify=df['category_id'],
    random_state=42
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train: {len(train_df)} rows')
print(f'Test : {len(test_df)} rows')
print(f'\nTest class distribution (top 5):')
print(test_df['category'].value_counts().head())

### 1.5 Shared Test Transform

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

print('Test transform:')
print(test_transform)

### 1.6 Dataset Classes

In [ ]:
class ProductDataset(Dataset):
    """Hierarchical category dataset → (image, main_label, sub_label)"""
    def __init__(self, df, image_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, str(row['category_id']),
                                f"{row['anonymous_id']}.jpg")
        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            # Category 44 images stored in original folder (before 44→28 merge)
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (255, 255, 255))
        if self.transform:
            image = self.transform(image)
        return image, int(row['main_encoded']), int(row['category_encoded'])


class MultitaskDataset(Dataset):
    """Multitask dataset → (image, main_label, sub_label, color_label, gender_label)"""
    def __init__(self, df, image_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, str(row['category_id']),
                                f"{row['anonymous_id']}.jpg")
        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (255, 255, 255))
        if self.transform:
            image = self.transform(image)
        return (
            image,
            int(row['main_encoded']),
            int(row['category_encoded']),
            int(row['color_encoded']),   # -1 if unknown
            int(row['gender_encoded']),
        )


class ColorDataset(Dataset):
    """Color-only dataset → (image, color_label). Skips rows with color='unknown'."""
    def __init__(self, df, image_dir, transform=None):
        # Keep only samples with known color
        self.df        = df[df['color_encoded'] != -1].reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, str(row['category_id']),
                                f"{row['anonymous_id']}.jpg")
        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (255, 255, 255))
        if self.transform:
            image = self.transform(image)
        return image, int(row['color_encoded'])


print('All three Dataset classes defined.')

### 1.7 DataLoaders & Batch Inspection

In [ ]:
# ── Test loaders (shuffle=False, no sampler) ──────────────────────────────────
test_product_loader = DataLoader(
    ProductDataset(test_df, IMAGE_DIR, test_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

test_multitask_loader = DataLoader(
    MultitaskDataset(test_df, IMAGE_DIR, test_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

test_color_loader = DataLoader(
    ColorDataset(test_df, IMAGE_DIR, test_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

# Inspect one batch from each loader
imgs, main_lbl, sub_lbl = next(iter(test_product_loader))
print(f'ProductDataset batch   — images: {tuple(imgs.shape)}  main: {tuple(main_lbl.shape)}  sub: {tuple(sub_lbl.shape)}')
print(f'  main label range: {main_lbl.min().item()} – {main_lbl.max().item()}')
print(f'  sub  label range: {sub_lbl.min().item()}  – {sub_lbl.max().item()}')

imgs_mt, ml, sl, cl, gl = next(iter(test_multitask_loader))
print(f'\nMultitaskDataset batch — images: {tuple(imgs_mt.shape)}')
print(f'  color label range: {cl.min().item()} (−1=unknown) – {cl.max().item()}')
print(f'  gender label range: {gl.min().item()} – {gl.max().item()}')

imgs_c, color_lbl = next(iter(test_color_loader))
print(f'\nColorDataset batch     — images: {tuple(imgs_c.shape)}')
print(f'  color label range: {color_lbl.min().item()} – {color_lbl.max().item()}')

print('\n[PASS] All DataLoaders working.')

---
## Part 2 — Model Inference

Each section: **define architecture → load `.pth` → run inference → report accuracy**.

No training — all cells just call `model.eval()` with `torch.no_grad()`.

### 2a. CLIP Hierarchical Model
Backbone: `openai/clip-vit-base-patch32` vision encoder (768-d).  
Heads: `main_head` (21 classes) → `sub_head` (concat + 190 classes).  
Checkpoint: `outputs/clip_hierarchical_best.pth`

In [ ]:
class CLIPHierarchicalModel(nn.Module):
    def __init__(self, num_main, num_sub, feature_dim=768):
        super().__init__()
        clip_base     = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
        self.backbone = clip_base.vision_model
        self.dropout  = nn.Dropout(0.3)
        self.main_head = nn.Sequential(
            nn.Linear(feature_dim, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_main)
        )
        self.sub_head = nn.Sequential(
            nn.Linear(feature_dim + num_main, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_sub)
        )

    def forward(self, x):
        features    = self.dropout(self.backbone(pixel_values=x).pooler_output)
        main_logits = self.main_head(features)
        sub_logits  = self.sub_head(torch.cat([features, main_logits], dim=1))
        return main_logits, sub_logits

print('CLIPHierarchicalModel class defined.')

In [ ]:
clip_hier_model = CLIPHierarchicalModel(NUM_MAIN, NUM_SUB)
state = torch.load(CLIP_HIER_PATH, map_location='cpu', weights_only=True)
clip_hier_model.load_state_dict(state)
clip_hier_model.to(device).eval()
n_params = sum(p.numel() for p in clip_hier_model.parameters())
print(f'Loaded {CLIP_HIER_PATH}')
print(f'Parameters: {n_params:,}')

In [ ]:
# Evaluate on test set
clip_hier_main_correct = clip_hier_sub_correct = clip_hier_total = 0
all_sub_preds_ch  = []
all_sub_labels_ch = []

with torch.no_grad():
    for imgs, main_lbl, sub_lbl in test_product_loader:
        imgs     = imgs.to(device)
        main_lbl = main_lbl.to(device)
        sub_lbl  = sub_lbl.to(device)
        main_logits, sub_logits = clip_hier_model(imgs)
        clip_hier_main_correct += (main_logits.argmax(1) == main_lbl).sum().item()
        clip_hier_sub_correct  += (sub_logits.argmax(1)  == sub_lbl).sum().item()
        clip_hier_total        += sub_lbl.size(0)
        all_sub_preds_ch.extend(sub_logits.argmax(1).cpu().numpy())
        all_sub_labels_ch.extend(sub_lbl.cpu().numpy())

clip_hier_main_acc = clip_hier_main_correct / clip_hier_total
clip_hier_sub_acc  = clip_hier_sub_correct  / clip_hier_total
clip_hier_f1       = f1_score(all_sub_labels_ch, all_sub_preds_ch, average='macro')

print(f'CLIP Hierarchical — Test Results')
print(f'  Main category  accuracy : {clip_hier_main_acc:.4f}')
print(f'  Subcategory    accuracy : {clip_hier_sub_acc:.4f}')
print(f'  Subcategory    F1 macro : {clip_hier_f1:.4f}')

### 2b. DINOv2-Base Hierarchical Model
Backbone: `facebookresearch/dinov2_vitb14` (768-d CLS token).  
Heads: same hierarchical structure as CLIP.  
Checkpoint: `outputs/dino_best.pth`

In [ ]:
class DINOv2HierarchicalModel(nn.Module):
    def __init__(self, num_main, num_sub, hidden=768):
        super().__init__()
        # pretrained=False because weights come from the .pth checkpoint
        self.backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14',
                                       pretrained=False)
        self.main_head = nn.Sequential(
            nn.Linear(hidden, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_main)
        )
        self.sub_head = nn.Sequential(
            nn.Linear(hidden + num_main, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, num_sub)
        )

    def forward(self, x):
        features    = self.backbone(x)           # (B, 768) CLS token
        main_logits = self.main_head(features)
        sub_logits  = self.sub_head(torch.cat([features, main_logits], dim=1))
        return main_logits, sub_logits

print('DINOv2HierarchicalModel class defined.')

In [ ]:
dino_model = DINOv2HierarchicalModel(NUM_MAIN, NUM_SUB)
state = torch.load(DINO_PATH, map_location='cpu', weights_only=True)
dino_model.load_state_dict(state)
dino_model.to(device).eval()
print(f'Loaded {DINO_PATH}')
print(f'Parameters: {sum(p.numel() for p in dino_model.parameters()):,}')

In [ ]:
dino_main_correct = dino_sub_correct = dino_total = 0
all_sub_preds_dino  = []
all_sub_labels_dino = []

with torch.no_grad():
    for imgs, main_lbl, sub_lbl in test_product_loader:
        imgs     = imgs.to(device)
        main_lbl = main_lbl.to(device)
        sub_lbl  = sub_lbl.to(device)
        main_logits, sub_logits = dino_model(imgs)
        dino_main_correct += (main_logits.argmax(1) == main_lbl).sum().item()
        dino_sub_correct  += (sub_logits.argmax(1)  == sub_lbl).sum().item()
        dino_total        += sub_lbl.size(0)
        all_sub_preds_dino.extend(sub_logits.argmax(1).cpu().numpy())
        all_sub_labels_dino.extend(sub_lbl.cpu().numpy())

dino_main_acc = dino_main_correct / dino_total
dino_sub_acc  = dino_sub_correct  / dino_total
dino_f1       = f1_score(all_sub_labels_dino, all_sub_preds_dino, average='macro')

print(f'DINOv2 Hierarchical — Test Results')
print(f'  Main category  accuracy : {dino_main_acc:.4f}')
print(f'  Subcategory    accuracy : {dino_sub_acc:.4f}')
print(f'  Subcategory    F1 macro : {dino_f1:.4f}')

### 2c. CLIP Multitask Model
Same CLIP backbone with 4 heads: main (21) → sub (190), color (12), gender (4).  
Gender head is conditioned on `features + main_logits + color_logits`.  
Checkpoint: `outputs/clip_multitask_best.pth`

In [ ]:
class CLIPMultitaskModel(nn.Module):
    def __init__(self, num_main, num_sub, num_colors, num_genders, feature_dim=768):
        super().__init__()
        clip_base     = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
        self.backbone = clip_base.vision_model
        self.dropout  = nn.Dropout(0.3)

        self.main_head = nn.Sequential(
            nn.Linear(feature_dim, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_main)
        )
        self.sub_head = nn.Sequential(
            nn.Linear(feature_dim + num_main, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_sub)
        )
        self.color_head = nn.Sequential(
            nn.Linear(feature_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_colors)
        )
        # gender conditioned on features + main_logits + color_logits
        self.gender_head = nn.Sequential(
            nn.Linear(feature_dim + num_main + num_colors, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_genders)
        )

    def forward(self, x):
        features     = self.dropout(self.backbone(pixel_values=x).pooler_output)
        main_logits  = self.main_head(features)
        sub_logits   = self.sub_head(torch.cat([features, main_logits], dim=1))
        color_logits = self.color_head(features)
        gender_logits= self.gender_head(torch.cat([features, main_logits, color_logits], dim=1))
        return main_logits, sub_logits, color_logits, gender_logits

print('CLIPMultitaskModel class defined.')

In [ ]:
clip_mt_model = CLIPMultitaskModel(NUM_MAIN, NUM_SUB, NUM_COLORS, NUM_GENDERS)
state = torch.load(CLIP_MT_PATH, map_location='cpu', weights_only=True)
clip_mt_model.load_state_dict(state)
clip_mt_model.to(device).eval()
print(f'Loaded {CLIP_MT_PATH}')
print(f'Parameters: {sum(p.numel() for p in clip_mt_model.parameters()):,}')

In [ ]:
mt_main_c = mt_sub_c = mt_color_c = mt_gender_c = mt_total = mt_color_total = 0
all_sub_preds_mt  = []
all_sub_labels_mt = []
all_color_preds_mt  = []
all_color_labels_mt = []
all_gen_preds_mt  = []
all_gen_labels_mt = []

with torch.no_grad():
    for imgs, ml, sl, cl, gl in test_multitask_loader:
        imgs = imgs.to(device)
        ml   = ml.to(device)
        sl   = sl.to(device)
        cl   = cl.to(device)
        gl   = gl.to(device)

        main_logits, sub_logits, color_logits, gender_logits = clip_mt_model(imgs)

        mt_main_c  += (main_logits.argmax(1) == ml).sum().item()
        mt_sub_c   += (sub_logits.argmax(1)  == sl).sum().item()
        mt_gender_c+= (gender_logits.argmax(1)== gl).sum().item()
        mt_total   += sl.size(0)

        color_mask = cl != -1
        if color_mask.sum() > 0:
            mt_color_c     += (color_logits[color_mask].argmax(1) == cl[color_mask]).sum().item()
            mt_color_total += color_mask.sum().item()
            all_color_preds_mt.extend(color_logits[color_mask].argmax(1).cpu().numpy())
            all_color_labels_mt.extend(cl[color_mask].cpu().numpy())

        all_sub_preds_mt.extend(sub_logits.argmax(1).cpu().numpy())
        all_sub_labels_mt.extend(sl.cpu().numpy())
        all_gen_preds_mt.extend(gender_logits.argmax(1).cpu().numpy())
        all_gen_labels_mt.extend(gl.cpu().numpy())

print('CLIP Multitask — Test Results')
print(f'  Main  accuracy : {mt_main_c / mt_total:.4f}')
print(f'  Sub   accuracy : {mt_sub_c  / mt_total:.4f}  |  F1 macro: {f1_score(all_sub_labels_mt,   all_sub_preds_mt,   average="macro"):.4f}')
print(f'  Color accuracy : {mt_color_c / mt_color_total:.4f}  |  F1 macro: {f1_score(all_color_labels_mt, all_color_preds_mt, average="macro"):.4f}')
print(f'  Gender accuracy: {mt_gender_c/ mt_total:.4f}  |  F1 macro: {f1_score(all_gen_labels_mt,   all_gen_preds_mt,   average="macro"):.4f}')

### 2d. EfficientNet-B3 Hierarchical Model
Backbone: `efficientnet_b3` (1536-d features after removing classifier).  
Heads: same hierarchical structure — main (21) + sub (190).  
Checkpoint: `EfficientNet/efficientnet_b3_best.pth`

In [ ]:
class EfficientNetHierarchicalModel(nn.Module):
    def __init__(self, num_main, num_sub):
        super().__init__()
        backbone     = models.efficientnet_b3(weights=None)
        feature_dim  = backbone.classifier[1].in_features  # 1536
        backbone.classifier = nn.Identity()
        self.backbone  = backbone
        self.dropout   = nn.Dropout(0.3)
        self.main_head = nn.Sequential(
            nn.Linear(feature_dim, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, num_main)
        )
        self.sub_head = nn.Sequential(
            nn.Linear(feature_dim + num_main, 512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, num_sub)
        )

    def forward(self, x):
        features    = self.dropout(self.backbone(x))
        main_logits = self.main_head(features)
        sub_logits  = self.sub_head(torch.cat([features, main_logits], dim=1))
        return main_logits, sub_logits

print('EfficientNetHierarchicalModel class defined.')

In [ ]:
effnet_model = EfficientNetHierarchicalModel(NUM_MAIN, NUM_SUB)
state = torch.load(EFFNET_PATH, map_location='cpu', weights_only=True)
effnet_model.load_state_dict(state)
effnet_model.to(device).eval()
print(f'Loaded {EFFNET_PATH}')
print(f'Parameters: {sum(p.numel() for p in effnet_model.parameters()):,}')

In [ ]:
effnet_main_c = effnet_sub_c = effnet_total = 0
all_sub_preds_en  = []
all_sub_labels_en = []

with torch.no_grad():
    for imgs, main_lbl, sub_lbl in test_product_loader:
        imgs     = imgs.to(device)
        main_lbl = main_lbl.to(device)
        sub_lbl  = sub_lbl.to(device)
        main_logits, sub_logits = effnet_model(imgs)
        effnet_main_c += (main_logits.argmax(1) == main_lbl).sum().item()
        effnet_sub_c  += (sub_logits.argmax(1)  == sub_lbl).sum().item()
        effnet_total  += sub_lbl.size(0)
        all_sub_preds_en.extend(sub_logits.argmax(1).cpu().numpy())
        all_sub_labels_en.extend(sub_lbl.cpu().numpy())

effnet_main_acc = effnet_main_c / effnet_total
effnet_sub_acc  = effnet_sub_c  / effnet_total
effnet_f1       = f1_score(all_sub_labels_en, all_sub_preds_en, average='macro')

print(f'EfficientNet-B3 Hierarchical — Test Results')
print(f'  Main category  accuracy : {effnet_main_acc:.4f}')
print(f'  Subcategory    accuracy : {effnet_sub_acc:.4f}')
print(f'  Subcategory    F1 macro : {effnet_f1:.4f}')

### 2e. ResNet18 Color Classifier
Backbone: `resnet18` with custom fc head: `512 → 256 → NUM_COLORS`.  
Only evaluated on samples with known color labels.  
Checkpoint: `outputs/color_resnet18_best.pth`

In [ ]:
def build_resnet18_color(num_colors):
    backbone = models.resnet18(weights=None)
    feature_dim = backbone.fc.in_features  # 512
    backbone.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(feature_dim, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, num_colors)
    )
    return backbone

color_model = build_resnet18_color(NUM_COLORS)
state = torch.load(COLOR_PATH, map_location='cpu', weights_only=True)
color_model.load_state_dict(state)
color_model.to(device).eval()
print(f'Loaded {COLOR_PATH}')
print(f'Parameters: {sum(p.numel() for p in color_model.parameters()):,}')

In [ ]:
color_correct = color_total = 0
all_preds_color  = []
all_labels_color = []

with torch.no_grad():
    for imgs, color_lbl in test_color_loader:
        imgs      = imgs.to(device)
        color_lbl = color_lbl.to(device)
        logits    = color_model(imgs)
        color_correct += (logits.argmax(1) == color_lbl).sum().item()
        color_total   += color_lbl.size(0)
        all_preds_color.extend(logits.argmax(1).cpu().numpy())
        all_labels_color.extend(color_lbl.cpu().numpy())

color_acc = color_correct / color_total
color_f1  = f1_score(all_labels_color, all_preds_color, average='macro')

print(f'ResNet18 Color — Test Results  ({color_total} samples with known color)')
print(f'  Accuracy : {color_acc:.4f}')
print(f'  F1 macro : {color_f1:.4f}')
print(f'\nPer-class report:')
print(classification_report(all_labels_color, all_preds_color,
                             target_names=color_encoder.classes_))

### 2f. Gender Logistic Regression
Features: OneHot(`level_2_id`, `category_id`, `color_label`).  
Model: sklearn `Pipeline` → `OneHotEncoder` → `LogisticRegression(class_weight='balanced')`.  
Artifacts: `outputs/gender_lr_model.pkl`, `gender_label_encoder.pkl`, `gender_feature_cols.pkl`

In [ ]:
with open(GENDER_LR_PATH,  'rb') as f: gender_lr_pipeline = pickle.load(f)
with open(GENDER_ENC_PATH, 'rb') as f: gender_le          = pickle.load(f)
with open(GENDER_COLS_PATH,'rb') as f: gender_feature_cols = pickle.load(f)

print('Gender LR pipeline loaded.')
print(f'Gender classes : {list(gender_le.classes_)}')
print(f'Feature columns: {gender_feature_cols}')

In [ ]:
# Build features (mirrors LogisticRegressionforgender.py)
test_gender_df = test_df.copy()
test_gender_df = test_gender_df.dropna(subset=['gender'])

if 'level_2_id' in test_gender_df.columns:
    test_gender_df['main_cat'] = test_gender_df['level_2_id'].astype(str)

test_gender_df['sub_cat']    = test_gender_df['category_id'].astype(str)
test_gender_df['color_feat'] = test_gender_df['color_label'].fillna('unknown').astype(str)

X_gender = test_gender_df[gender_feature_cols]
y_gender = gender_le.transform(test_gender_df['gender'])

y_pred_gender = gender_lr_pipeline.predict(X_gender)

gender_acc = (y_pred_gender == y_gender).mean()
gender_f1  = f1_score(y_gender, y_pred_gender, average='macro')

print(f'Gender LR — Test Results  ({len(y_gender)} samples)')
print(f'  Accuracy : {gender_acc:.4f}')
print(f'  F1 macro : {gender_f1:.4f}')
print(f'\nPer-class report:')
print(classification_report(y_gender, y_pred_gender, target_names=gender_le.classes_))

---
## Part 3 — CLIP + DINOv2 Ensemble

Weighted average of subcategory softmax probabilities.  
Default weights: CLIP 0.6, DINO 0.4 (configurable below).

In [ ]:
CLIP_WEIGHT = 0.6
DINO_WEIGHT = 0.4

ens_sub_correct = ens_total = 0
all_sub_preds_ens  = []
all_sub_labels_ens = []

with torch.no_grad():
    for imgs, main_lbl, sub_lbl in test_product_loader:
        imgs    = imgs.to(device)
        sub_lbl = sub_lbl.to(device)

        _, clip_sub_logits  = clip_hier_model(imgs)
        _, dino_sub_logits  = dino_model(imgs)

        clip_probs = F.softmax(clip_sub_logits, dim=1)
        dino_probs = F.softmax(dino_sub_logits, dim=1)

        ens_probs = CLIP_WEIGHT * clip_probs + DINO_WEIGHT * dino_probs
        preds     = ens_probs.argmax(dim=1)

        ens_sub_correct += (preds == sub_lbl).sum().item()
        ens_total       += sub_lbl.size(0)
        all_sub_preds_ens.extend(preds.cpu().numpy())
        all_sub_labels_ens.extend(sub_lbl.cpu().numpy())

ens_acc = ens_sub_correct / ens_total
ens_f1  = f1_score(all_sub_labels_ens, all_sub_preds_ens, average='macro')

print(f'Ensemble (CLIP {CLIP_WEIGHT} + DINO {DINO_WEIGHT}) — Test Results')
print(f'  Subcategory accuracy : {ens_acc:.4f}')
print(f'  Subcategory F1 macro : {ens_f1:.4f}')

---
## Part 4 — Results Summary

In [ ]:
results = pd.DataFrame([
    {
        'Model':            'CLIP Hierarchical',
        'Task':             'Category',
        'Main Acc':         f'{clip_hier_main_acc:.4f}',
        'Sub Acc':          f'{clip_hier_sub_acc:.4f}',
        'Sub F1 macro':     f'{clip_hier_f1:.4f}',
        'Color Acc':        '—',
        'Gender Acc':       '—',
    },
    {
        'Model':            'DINOv2 Hierarchical',
        'Task':             'Category',
        'Main Acc':         f'{dino_main_acc:.4f}',
        'Sub Acc':          f'{dino_sub_acc:.4f}',
        'Sub F1 macro':     f'{dino_f1:.4f}',
        'Color Acc':        '—',
        'Gender Acc':       '—',
    },
    {
        'Model':            'CLIP + DINOv2 Ensemble',
        'Task':             'Category',
        'Main Acc':         '—',
        'Sub Acc':          f'{ens_acc:.4f}',
        'Sub F1 macro':     f'{ens_f1:.4f}',
        'Color Acc':        '—',
        'Gender Acc':       '—',
    },
    {
        'Model':            'CLIP Multitask',
        'Task':             'Category + Color + Gender',
        'Main Acc':         f'{mt_main_c / mt_total:.4f}',
        'Sub Acc':          f'{mt_sub_c  / mt_total:.4f}',
        'Sub F1 macro':     f'{f1_score(all_sub_labels_mt, all_sub_preds_mt, average="macro"):.4f}',
        'Color Acc':        f'{mt_color_c / mt_color_total:.4f}',
        'Gender Acc':       f'{mt_gender_c / mt_total:.4f}',
    },
    {
        'Model':            'EfficientNet-B3 Hierarchical',
        'Task':             'Category',
        'Main Acc':         f'{effnet_main_acc:.4f}',
        'Sub Acc':          f'{effnet_sub_acc:.4f}',
        'Sub F1 macro':     f'{effnet_f1:.4f}',
        'Color Acc':        '—',
        'Gender Acc':       '—',
    },
    {
        'Model':            'ResNet18 Color',
        'Task':             'Color (12 classes)',
        'Main Acc':         '—',
        'Sub Acc':          '—',
        'Sub F1 macro':     '—',
        'Color Acc':        f'{color_acc:.4f}',
        'Gender Acc':       '—',
    },
    {
        'Model':            'Logistic Regression (Gender)',
        'Task':             'Gender (4 classes)',
        'Main Acc':         '—',
        'Sub Acc':          '—',
        'Sub F1 macro':     '—',
        'Color Acc':        '—',
        'Gender Acc':       f'{gender_acc:.4f}  (F1={gender_f1:.4f})',
    },
])

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 140)
print(results.to_string(index=False))